# 01. Relational Data Extraction (SQL) & Exploratory Data Analysis (EDA)
**Objective:** Inspect raw credit risk records, normalize data into a local SQLite relational database, and execute SQL queries to extract consolidated borrower features.

In [1]:
from pathlib import Path
import pandas as pd

In [ ]:
# Define robust relative path to the data directory
# Hint: Since this notebook is inside 'notebooks/', the project root is one level up ('..')
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_PATH = PROJECT_ROOT / "data" / "credit_risk_dataset.csv"

# Load the CSV file into a pandas DataFrame named 'df_raw'
df_raw = pd.read_csv(RAW_DATA_PATH)

In [ ]:
# Print the shape of the dataset (rows, columns) and verify data types (.info() or .dtypes)
print(df_raw.shape)
print(df_raw.info())

(32581, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB
None


In [ ]:
# Calculate and print the exact number AND percentage of missing values (NaN) per column
# print(pd.isna(df_raw).sum(), '\n', pd.isna(df_raw).sum() / len(df_raw) * 100)
null_counts = pd.isna(df_raw).sum()
print(pd.DataFrame({
    'Total Rows': df_raw.count(),
    'Exact Missing': null_counts,
    'Percentage (%)': (null_counts / len(df_raw) * 100).round(1)
}))


                            Total Rows  Exact Missing  Percentage (%)
person_age                       32581              0             0.0
person_income                    32581              0             0.0
person_home_ownership            32581              0             0.0
person_emp_length                31686            895             2.7
loan_intent                      32581              0             0.0
loan_grade                       32581              0             0.0
loan_amnt                        32581              0             0.0
loan_int_rate                    29465           3116             9.6
loan_status                      32581              0             0.0
loan_percent_income              32581              0             0.0
cb_person_default_on_file        32581              0             0.0
cb_person_cred_hist_length       32581              0             0.0


In [ ]:
# Inspect statistical summary of numerical columns (.describe()) 
df_raw.describe()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length
count,32581.000000,3.258100e+04,31686.000000,32581.000000,29465.000000,32581.000000,32581.000000,32581.000000
mean,27.734600,6.607485e+04,4.789686,9589.371106,11.011695,0.218164,0.170203,5.804211
std,6.348078,6.198312e+04,4.142630,6322.086646,3.240459,0.413006,0.106782,4.055001
min,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000
25%,23.000000,3.850000e+04,2.000000,5000.000000,7.900000,0.000000,0.090000,3.000000
50%,26.000000,5.500000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000
75%,30.000000,7.920000e+04,7.000000,12200.000000,13.470000,0.000000,0.230000,8.000000
max,144.000000,6.000000e+06,123.000000,35000.000000,23.220000,1.000000,0.830000,30.000000


In [ ]:
# Looking closely at the 'max' values of 'person_age' and 'person_emp_length'
# They don't make a physical/human sense

import sqlite3

In [ ]:
# Create a unique identifier for each row
df_raw['applicant_id'] = range(1, len(df_raw) + 1)

# Create 'df_applicants' containing only demographic and credit history columns.
df_applicants = df_raw[['applicant_id', 'person_age', 'person_income', 'person_home_ownership', 
                'person_emp_length', 'cb_person_default_on_file', 'cb_person_cred_hist_length']].copy()


# Create 'df_loans' containing only loan-specific columns.
df_loans = df_raw[['applicant_id', 'loan_intent', 'loan_grade', 'loan_amnt', 
                'loan_int_rate', 'loan_status', 'loan_percent_income']].copy()


# Verify the split
print(f"Applicants table shape: {df_applicants.shape}")
print(f"Loans table shape: {df_loans.shape}")

# Connect to SQLite and write these DataFrames as SQL tables.
# Define the DB path
DB_PATH = PROJECT_ROOT / "data" / "credit_risk.db"

# Create connection -> Use to_sql() for both dataframes -> Close the connection
conn = sqlite3.connect(DB_PATH)
df_applicants.to_sql(name="applicants", con=conn, if_exists="replace", index=False)
df_loans.to_sql(name="loans", con=conn, if_exists="replace", index=False)
conn.close()


print("Database created successfully at data/credit_risk.db")

Applicants table shape: (32581, 7)
Loans table shape: (32581, 7)
Database created successfully at data/credit_risk.db


In [55]:
# Execute the SQL query from the .sql file to create the working DataFrame
DB_PATH = PROJECT_ROOT / "data" / "credit_risk.db"
QUERY_PATH = PROJECT_ROOT / "src" / "queries.sql"

# Read the raw SQL text
sql_query = QUERY_PATH.read_text()

# Connect and extract
with sqlite3.connect(DB_PATH) as conn:
    df_consolidated = pd.read_sql_query(sql_query, conn)

print(df_consolidated.head())
print(f"\nConsolidated shape: {df_consolidated.shape}")

   applicant_id  person_age  person_income  person_emp_length  loan_amnt  \
0             1          22          59000              123.0      35000   
1             2          21           9600                5.0       1000   
2             3          25           9600                1.0       5500   
3             4          23          65500                4.0      35000   
4             5          24          54400                8.0      35000   

   loan_int_rate  loan_status  
0          16.02            1  
1          11.14            0  
2          12.87            1  
3          15.23            1  
4          14.27            1  

Consolidated shape: (32576, 7)
